# Structure-Aware RAG for AI Research Papers

End-to-end pipeline comparing **structure-aware chunking** (tables, figures, and captions as atomic retrieval units) against **standard fixed-size chunking** on a hand-built QA benchmark over ~30 RAG research papers.

**Sequence:**
1. **Module 1 — Parsing & Correction**: PDFs → structured block JSONs, manual correction pass, corpus quality verification.
2. **Module 2 — Benchmarking, Retrieval & Evaluation**: QA benchmark generation/verification, fixed-size vs. structure-aware chunking + dual Chroma indexing, retrieval scoring (Recall@k / MRR), and end-to-end evaluation (LLM-as-judge, statistical significance).

Pipeline logic lives in versioned scripts under `scripts/` (`parse_papers.py`, `correction_helper.py`, `generate_qa_pairs.py`, `fix_manual_transcription_flags.py`, `verify_qa_pairs.py`) added separately and is called/imported here rather than pasted inline, so it stays diffable and testable outside Colab.


## Environment setup


In [ ]:
!apt-get install -y ghostscript
!pip install -q pymupdf pdfplumber camelot-py opencv-python-headless requests chromadb google-genai

from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/RAG

import os
import sys
from google.colab import userdata

pdf_dir = "/content/drive/MyDrive/RAG/rag_papers"
SCRIPTS_DIR = "/content/drive/MyDrive/RAG/scripts"
sys.path.append(SCRIPTS_DIR)

os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

print(os.listdir(pdf_dir))


## Module 1: Parsing & Correction
Parses the raw PDF corpus into structured block JSONs (`{paper_id, blocks, flagged_items}`), runs a manual correction pass on flagged caption/object links, and verifies corpus quality before handing off to benchmark construction.

### Parsing pipeline
Import the parsing pipeline from `parse_papers.py` rather than defining it inline, so the logic stays version-controlled and reviewable as a standalone script.


In [ ]:
import sys
SCRIPTS_DIR = "/content/drive/MyDrive/RAG/scripts"
sys.path.append(SCRIPTS_DIR)

from parse_papers import (
    parse_paper,
    batch_parse_papers,
    extract_tables,
    extract_figures,
    build_blocks,
    summarize_parse_quality,
)

### Run the batch parse
Corpus manifest (PDF path -> arXiv ID) and the call that parses all 30 papers into `parsed_output/`.

In [ ]:
papers = [
    (os.path.join(pdf_dir, "asai_self_rag.pdf"), "2310.11511"),
    (os.path.join(pdf_dir, "chen_rgb.pdf"), "2309.01431"),
    (os.path.join(pdf_dir, "cheng_coral.pdf"), "2410.23090"),
    (os.path.join(pdf_dir, "es_ragas.pdf"), "2309.15217"),
    (os.path.join(pdf_dir, "fan_rag_llm.pdf"), "2405.06211"),
    (os.path.join(pdf_dir, "friel_rag_bench.pdf"), "2407.11005"),
    (os.path.join(pdf_dir, "gao_rag_survey.pdf"), "2312.10997"),
    (os.path.join(pdf_dir, "izacard_atlas.pdf"), "2208.03299"),
    (os.path.join(pdf_dir, "izacard_odq.pdf"), "2007.01282"),
    (os.path.join(pdf_dir, "jeong_adaptive_rag.pdf"), "2403.14403"),
    (os.path.join(pdf_dir, "jiang_active_rag.pdf"), "2305.06983"),
    (os.path.join(pdf_dir, "lewis_rag.pdf"), "2005.11401"),
    (os.path.join(pdf_dir, "lin_ra_dit.pdf"), "2310.01352"),
    (os.path.join(pdf_dir, "liu_recall.pdf"), "2311.08147"),
    (os.path.join(pdf_dir, "lyu_crud_rag.pdf"), "2401.17043"),
    (os.path.join(pdf_dir, "maiorano_readiness.pdf"), "2603.27355"),
    (os.path.join(pdf_dir, "rackauckas_rag_fusion.pdf"), "2402.03367"),
    (os.path.join(pdf_dir, "saad_falcon_ares.pdf"), "2311.09476"),
    (os.path.join(pdf_dir, "sarthi_raptor.pdf"), "2401.18059"),
    (os.path.join(pdf_dir, "sharma_rag_survey.pdf"), "2506.00054"),
    (os.path.join(pdf_dir, "shi_replug.pdf"), "2301.12652"),
    (os.path.join(pdf_dir, "song_query_optim.pdf"), "2412.17558"),
    (os.path.join(pdf_dir, "xu_recomp.pdf"), "2310.04408"),
    (os.path.join(pdf_dir, "yan_crag.pdf"), "2401.15884"),
    (os.path.join(pdf_dir, "yang_crag_bench.pdf"), "2406.04744"),
    (os.path.join(pdf_dir, "yang_multi_hop.pdf"), "2401.15391"),
    (os.path.join(pdf_dir, "yu_chain_of_note.pdf"), "2311.09210"),
    (os.path.join(pdf_dir, "zhang_raft.pdf"), "2403.10131"),
    (os.path.join(pdf_dir, "zhang_rag_poisoning.pdf"), "2505.18543"),
    (os.path.join(pdf_dir, "zheng_step_back.pdf"), "2310.06117"),
]

result = batch_parse_papers(papers, output_dir="parsed_output")

### Correction workflow
Import the review/correction helpers from `correction_helper.py`. These operate on the parsed JSONs in place (`load_paper` / `save_paper`), so re-running this cell doesn't repeat any correction — it just makes the functions available.

In [ ]:
from correction_helper import (
    load_paper,
    save_paper,
    show_flagged,
    show_all_flagged,
    apply_fix,
    add_missing_table,
    add_missing_figure,
)

### Review flagged caption/object links
Papers below had at least one caption the parser couldn't confidently link to a table or figure block. `RAPTOR` appendix Tables 9 and 14–21 are a known, permanent exception (no caption blocks were ever detected for them) and are excluded from QA generation entirely — not fixed here.

In [ ]:
show_all_flagged([
    "2309.01431", "2405.06211", "2312.10997", "2208.03299", "2403.14403",
    "2305.06983", "2401.17043", "2603.27355", "2311.09476", "2401.18059",
    "2301.12652", "2310.04408", "2401.15884", "2406.04744", "2401.15391",
    "2311.09210", "2403.10131", "2505.18543",
])

### Apply manual corrections
Hand-transcribed tables/figures for captions the parser located but couldn't extract structured content for (dense two-column IEEE layouts, borderless tables — see the Camelot limitation note below). These six papers are flagged via `manual_transcription_flag` for separate downstream analysis: RGB (2309.01431), Active RAG (2305.06983), Chain-of-Note (2311.09210), RAFT (2403.10131), RAG poisoning (2505.18543), Gao survey (2312.10997).

In [ ]:
add_missing_table(
    "2309.01431", page=6,
    rows=[
        ["Languages", "English", "", "Chinese", ""],
        ["", "Rej", "Rej*", "Rej", "Rej*"],
        ["ChatGPT", "24.67", "45.00", "5.33", "43.33"],
        ["ChatGLM-6B", "9.00", "25.00", "6.33", "17.00"],
        ["ChatGLM2-6B", "10.33", "41.33", "6.33", "36.33"],
        ["Vicuna-7B-v1.3", "17.00", "33.33", "3.37", "24.67"],
        ["Qwen-7B-Chat", "31.00", "35.67", "8.67", "25.33"],
        ["BELLE-7B-2M", "5.67", "32.33", "5.33", "13.67"],
    ],
    caption_block_id="2309.01431_p006_b0164",
)

add_missing_table(
    "2309.01431", page=7,
    rows=[
        ["", "English", "", "", "Chinese", "", ""],
        ["Noise Ratio", "0", "0.2", "0.4", "0", "0.2", "0.4"],
        ["ChatGPT", "55", "51", "34", "63", "58", "47"],
        ["ChatGLM-6B", "45", "36", "35", "60", "53", "52"],
        ["ChatGLM2-6B", "34", "32", "21", "44", "43", "32"],
        ["Vicuna-7B-v1.3", "60", "53", "43", "43", "36", "25"],
        ["Qwen-7B-Chat", "55", "50", "37", "67", "56", "55"],
        ["BELLE-7B-2M", "40", "34", "24", "49", "41", "38"],
    ],
    caption_block_id="2309.01431_p007_b0178",
)

add_missing_table(
    "2309.01431", page=7,
    rows=[
        ["", "Acc", "Acc_doc", "ED", "ED*", "CR"],
        ["ChatGPT-zh", "91", "17", "1", "3", "33.33"],
        ["Qwen-7B-Chat-zh", "77", "12", "5", "4", "25.00"],
        ["ChatGPT-en", "89", "9", "8", "7", "57.14"],
    ],
    caption_block_id="2309.01431_p007_b0202",
)

add_missing_figure(
    "2208.03299", page=3,
    caption_block_id="2208.03299_p003_b0049",
    content=[
        ["Task", "Query", "Output"],
        ["Fact Checking", "Bermuda Triangle is in the western part of the Himalayas.", "False"],
        ["Question Answering", "who is playing the halftime show at super bowl 2016", "Coldplay"],
        ["Entity Linking", "NTFS-3G is an open source <E>cross-platform</E> implementation of the Microsoft Windows NTFS file system with read-write support.", "Cross-platform software"],
    ],
)

add_missing_table(
    "2208.03299", page=13,
    rows=[
        ["", "Zero-shot", "5-shot", "5-shot (multi-task)", "Full / Transfer"],
        ["Standard Inference", "36.8", "43.4", "56.4", "65.8"],
        ["De-biased Inference", "47.1", "47.9", "56.6", "66.0"],
    ],
    caption_block_id="2208.03299_p013_b0200",
)

add_missing_table(
    "2403.14403", page=8,
    rows=[
        ["Training Strategies", "F1", "Step", "All", "No", "One", "Multi"],
        ["Adaptive-RAG (Ours)", "46.94", "1084", "54.52", "30.52", "66.28", "65.45"],
        ["w/o Binary", "43.43", "640", "60.30", "62.19", "65.70", "39.55"],
        ["w/o Silver", "48.79", "1464", "40.00", "0.00", "53.98", "75.91"],
    ],
    caption_block_id="2403.14403_p008_b0187",
)

add_missing_table(
    "2305.06983", page=9,
    rows=[
        ["β", "EM", "F1", "Prec.", "Rec."],
        ["0.0", "0.488", "0.576", "0.571", "0.605"],
        ["0.2", "0.498", "0.588", "0.582", "0.616"],
        ["0.4", "0.510", "0.597", "0.591", "0.627"],
        ["0.6", "0.506", "0.593", "0.586", "0.622"],
    ],
    caption_block_id="2305.06983_p009_b0211",
)

add_missing_table(
    "2305.06983", page=9,
    rows=[
        ["", "ASQA-hint EM", "ASQA-hint D-F1", "ASQA-hint R-L", "ASQA-hint DR", "WikiAsp UniEval", "WikiAsp E-F1", "WikiAsp R-L"],
        ["Implicit", "45.7", "36.9", "37.7", "37.3", "53.4", "18.8", "27.7"],
        ["Explicit", "46.2", "36.7", "37.7", "37.2", "53.4", "18.9", "27.6"],
    ],
    caption_block_id="2305.06983_p009_b0214",
)

add_missing_table(
    "2401.18059", page=7,
    rows=[
        ["Model", "ROUGE", "BLEU-1", "BLEU-4", "METEOR"],
        ["SBERT with RAPTOR", "30.87%", "23.50%", "6.42%", "19.20%"],
        ["SBERT without RAPTOR", "29.26%", "22.56%", "5.95%", "18.15%"],
        ["BM25 with RAPTOR", "27.93%", "21.17%", "5.70%", "17.03%"],
        ["BM25 without RAPTOR", "23.52%", "17.73%", "4.65%", "13.98%"],
        ["DPR with RAPTOR", "30.94%", "23.51%", "6.45%", "19.05%"],
        ["DPR without RAPTOR", "29.56%", "22.84%", "6.12%", "18.44%"],
    ],
    caption_block_id="2401.18059_p007_b0115",
)

add_missing_table(
    "2310.04408", page=8,
    rows=[
        ["Evidence", "EM", "% Gold in Evi.", "% Pred in Evi."],
        ["Top 1", "33.1", "36", "92 / 51"],
        ["Top 5", "39.3", "57", "96 / 81"],
        ["NE", "26.0", "46", "84 / 48"],
        ["Oracle sent", "60.2", "34", "93 / 16"],
        ["Contriever", "30.2", "25", "88 / 36"],
        ["Ours", "36.6", "28", "90 / 33"],
        ["GPT-3.5", "37.1", "45", "98 / 85"],
        ["T5", "25.9", "30", "52 / 20"],
        ["Ours", "37.0", "34", "98 / 39"],
    ],
    caption_block_id="2310.04408_p008_b0126",
)

add_missing_table(
    "2310.04408", page=9,
    rows=[
        ["Dataset", "Model", "% Faithful Y", "% Faithful P", "% Faithful N", "% Compre. Y", "% Compre. P", "% Compre. N", "% Use."],
        ["NQ", "GPT-3.5", "90", "0", "10", "97", "0", "3", "83"],
        ["NQ", "Ours", "80", "13", "7", "100", "0", "0", "80"],
        ["TQA", "GPT-3.5", "97", "0", "3", "90", "0", "10", "83"],
        ["TQA", "Ours", "83", "3", "14", "96", "0", "4", "77"],
        ["HQA", "GPT-3.5", "74", "0", "26", "78", "0", "22", "50"],
        ["HQA", "Ours", "67", "0", "33", "74", "0", "26", "40"],
    ],
    caption_block_id="2310.04408_p009_b0136",
)

add_missing_table(
    "2406.04744", page=4,
    rows=[
        ["Question type", "Definition"],
        ["Simple", "Questions asking for simple facts that are unlikely to change overtime, such as the birth date of a person and the authors of a book."],
        ["Simple w. Condition", "Questions asking for simple facts with some given conditions, such as stock prices on a certain date and a director's recent movies in a certain genre."],
        ["Set", "Questions that expect a set of entities or objects as the answer (e.g., \"what are the continents in the southern hemisphere?\")."],
        ["Comparison", "Questions that compare two entities (e.g., \"who started performing earlier, Adele or Ed Sheeran?\")."],
        ["Aggregation", "Questions that require aggregation of retrieval results to answer (e.g., \"how many Oscar awards did Meryl Streep win?\")."],
        ["Multi-hop", "Questions that require chaining multiple pieces of information to compose the answer (e.g., \"who acted in Ang Lee's latest movie?\")."],
        ["Post-processing heavy", "Questions that need reasoning or processing of the retrieved information to obtain the answer (e.g., \"how many days did Thurgood Marshall serve as a Supreme Court justice?\")."],
        ["False Premise", "Questions that have a false preposition or assumption (e.g., \"What's the name of Taylor Swift's rap album before she transitioned to pop?\" (Taylor Swift has not yet released any rap album))."],
    ],
    caption_block_id="2406.04744_p004_b0066",
)

add_missing_table(
    "2401.15391", page=2,
    rows=[
        ["News source", "Fortune Magazine", "The Sydney Morning Herald"],
        ["Evidence", "Back then, just like today, home prices had boomed for years before Fed officials were ultimately forced to hike interest rates aggressively in an attempt to fight inflation.", "Postponements of such reports could complicate things for the Fed, which has insisted it will make upcoming decisions on interest rates based on what incoming data say about the economy."],
        ["Claim", "Federal Reserve officials were forced to aggressively hike interest rates to combat inflation after years of booming home prices.", "The Federal Reserve has insisted that it will base its upcoming decisions on interest rates on the incoming economic data."],
        ["Bridge-Topic", "Interest rate hikes to combat inflation", "Interest rate decisions based on economic data"],
        ["Bridge-Entity", "Federal Reserve", "Federal Reserve"],
        ["Query", "Does the article from Fortune suggest that the Federal Reserve's interest rate hikes are a response to past conditions, such as booming home prices, while The Sydney Morning Herald article indicates that the Federal Reserve's future interest rate decisions will be based on incoming economic data?", ""],
        ["Answer", "Yes", ""],
    ],
    caption_block_id="2401.15391_p002_b0017",
)

add_missing_table(
    "2401.15391", page=6,
    rows=[
        ["Num. of Evidence Needed", "Count", "Percentage"],
        ["0 (Null Query)", "301", "11.78%"],
        ["2", "1078", "42.18%"],
        ["3", "779", "30.48%"],
        ["4", "398", "15.56%"],
        ["Total", "2,556", "100.00%"],
    ],
    caption_block_id="2401.15391_p006_b0062",
)

add_missing_table(
    "2403.10131", page=5,
    rows=[
        ["", "PubMed", "HotPot", "HuggingFace", "Torch Hub", "TensorFlow"],
        ["GPT-3.5 + RAG", "71.60", "41.5", "29.08", "60.21", "65.59"],
        ["LLaMA2-7B", "56.5", "0.54", "0.22", "0", "0"],
        ["LLaMA2-7B + RAG", "58.8", "0.03", "26.43", "08.60", "43.06"],
        ["DSF", "59.7", "6.38", "61.06", "84.94", "86.56"],
        ["DSF + RAG", "71.6", "4.41", "42.59", "82.80", "60.29"],
        ["RAFT (LLaMA2-7B)", "73.30", "35.28", "74.00", "84.95", "86.86"],
    ],
    caption_block_id="2403.10131_p005_b0069",
)

add_missing_table(
    "2505.18543", page=4,
    rows=[
        ["Category", "Attack", "Knowledge database", "Retriever", "LLM", "Targeted query"],
        ["Targeted poisoning", "BPRAG", "✗", "✗", "✗", "✓"],
        ["Targeted poisoning", "WPRAG", "✗", "✓", "✗", "✓"],
        ["Targeted poisoning", "BPI", "✗", "✗", "✗", "✓"],
        ["Targeted poisoning", "WPI", "✗", "✓", "✗", "✓"],
        ["Targeted poisoning", "AGGD", "✗", "✓", "✗", "✓"],
        ["Targeted poisoning", "CRAG-AS", "✗", "✗", "✗", "✓"],
        ["Targeted poisoning", "CRAG-AK", "✗", "✗", "✗", "✓"],
        ["DoS", "JamInject", "✗", "✗", "✗", "✓"],
        ["DoS", "JamOracle", "✗", "✗", "✗", "✓"],
        ["DoS", "JamOpt", "✗", "✗", "✗", "✓"],
        ["Trigger-based DoS", "AP", "✓", "✓", "✗", "✗"],
        ["Trigger-based DoS", "BadRAG", "✗", "✓", "✗", "✗"],
        ["Trigger-based DoS", "Phantom", "✗", "✓", "✗", "✗"],
    ],
    caption_block_id="2505.18543_p004_b0051",
)

add_missing_table(
    "2401.15884", page=8,
    rows=[
        ["", "LLaMA2-hf-7b", "SelfRAG-LLaMA2-7b"],
        ["CRAG", "54.9", "59.8"],
        ["w/o. Correct", "53.2", "58.3"],
        ["w/o. Incorrect", "54.4", "59.5"],
        ["w/o. Ambiguous", "54.0", "59.0"],
        ["Self-CRAG", "49.0", "61.8"],
        ["w/o. Correct", "43.6", "59.6"],
        ["w/o. Incorrect", "47.7", "60.8"],
        ["w/o. Ambiguous", "48.1", "61.5"],
    ],
    caption_block_id="2401.15884_p008_b0144",
)

add_missing_table(
    "2401.15884", page=8,
    rows=[
        ["", "LLaMA2-hf-7b", "SelfRAG-LLaMA2-7b"],
        ["CRAG", "54.9", "59.8"],
        ["w/o. refinement", "49.8", "54.2"],
        ["w/o. rewriting", "51.7", "56.2"],
        ["w/o. selection", "50.9", "58.6"],
        ["Self-CRAG", "49.0", "61.8"],
        ["w/o. refinement", "35.9", "52.2"],
        ["w/o. rewriting", "37.2", "58.4"],
        ["w/o. selection", "24.9", "57.9"],
    ],
    caption_block_id="2401.15884_p008_b0323",
)

add_missing_table(
    "2311.09210", page=5,
    rows=[
        ["Datasets", "Full size", "IR Recall", "Subset size"],
        ["NQ", "3,610", "73.82", "2,086"],
        ["TriviaQA", "7,993", "89.95", "7,074"],
        ["WebQ", "2,032", "64.22", "1,231"],
    ],
    caption_block_id="2311.09210_p005_b0075",
)

add_missing_table(
    "2311.09210", page=7,
    rows=[
        ["Models", "RealTimeQA EM", "RealTimeQA F1", "RealTimeQA RR"],
        ["Retrieve-Read (Shi et al., 2023c)", "15.6", "19.9", "6.1"],
        ["+ Chain-of-Note (ours)", "15.7", "20.3", "13.0"],
    ],
    caption_block_id="2311.09210_p007_b0122",
)

add_missing_table(
    "2311.09210", page=7,
    rows=[
        ["Model", "Noise Ratio", "NQ EM", "NQ F1", "TriviaQA EM", "TriviaQA F1", "WebQ EM", "WebQ F1", "Average EM", "Average F1"],
        ["Retrieve-Read", "100%", "34.28", "41.74", "55.30", "61.67", "29.58", "46.34", "39.72", "49.92"],
        ["+ Chain-of-Note", "100%", "41.83", "49.58", "64.30", "70.00", "36.85", "53.07", "47.66", "57.55"],
        ["Δ", "100%", "+7.55", "+7.84", "+9.00", "+8.33", "+7.27", "+6.73", "+7.94", "+7.63"],
        ["Retrieve-Read", "80%", "54.28", "61.03", "73.83", "80.02", "35.46", "52.70", "54.52", "64.58"],
        ["+ Chain-of-Note", "80%", "56.63", "63.23", "75.89", "81.24", "40.60", "56.54", "57.70", "67.00"],
        ["Δ", "80%", "+2.35", "+2.20", "+2.06", "+1.22", "+5.14", "+3.84", "+3.18", "+2.42"],
        ["Retrieve-Read", "60%", "61.44", "67.94", "78.44", "83.65", "37.01", "54.16", "58.96", "68.58"],
        ["+ Chain-of-Note", "60%", "63.43", "69.33", "78.79", "84.07", "41.26", "56.91", "61.16", "70.10"],
        ["Δ", "60%", "+1.99", "+1.39", "+0.35", "+0.42", "+4.25", "+2.75", "+2.20", "+1.52"],
        ["Retrieve-Read", "40%", "64.62", "71.12", "80.56", "86.76", "38.40", "55.60", "61.19", "71.16"],
        ["+ Chain-of-Note", "40%", "65.91", "72.22", "81.72", "87.11", "42.16", "58.15", "63.26", "72.49"],
        ["Δ", "40%", "+1.29", "+1.10", "+1.16", "+0.35", "+3.76", "+2.55", "+2.07", "+1.33"],
        ["Retrieve-Read", "20%", "67.21", "73.69", "81.73", "87.89", "39.95", "56.66", "62.96", "72.75"],
        ["+ Chain-of-Note", "20%", "70.00", "76.08", "82.86", "88.24", "44.36", "60.13", "65.74", "74.82"],
        ["Δ", "20%", "+2.79", "+2.39", "+1.13", "+0.35", "+4.41", "+3.47", "+2.78", "+2.07"],
        ["Retrieve-Read", "0%", "69.23", "75.57", "83.34", "89.44", "42.24", "58.59", "64.93", "74.53"],
        ["+ Chain-of-Note", "0%", "73.28", "79.86", "83.52", "88.94", "46.16", "62.38", "67.65", "77.06"],
        ["Δ", "0%", "+4.05", "+4.29", "+0.18", "-0.50", "+3.92", "+3.79", "+2.72", "+2.53"],
    ],
    caption_block_id="2311.09210_p007_b0119",
)

add_missing_table(
    "2401.18059", page=8,
    rows=[
        ["Model", "Accuracy (QuALITY)", "Answer F1 (QASPER)"],
        ["SBERT with RAPTOR", "56.6%", "36.70%"],
        ["SBERT without RAPTOR", "54.9%", "36.23%"],
        ["BM25 with RAPTOR", "52.1%", "27.00%"],
        ["BM25 without RAPTOR", "49.9%", "26.47%"],
        ["DPR with RAPTOR", "54.7%", "32.23%"],
        ["DPR without RAPTOR", "53.1%", "31.70%"],
    ],
    caption_block_id="2401.18059_p008_b0123",
)

add_missing_table(
    "2401.18059", page=8,
    rows=[
        ["Retriever", "GPT-3 F-1 Match", "GPT-4 F-1 Match", "UnifiedQA F-1 Match"],
        ["Title + Abstract", "25.2", "22.2", "17.5"],
        ["BM25", "46.6", "50.2", "26.4"],
        ["DPR", "51.3", "53.0", "32.1"],
        ["RAPTOR", "53.1", "55.7", "36.6"],
    ],
    caption_block_id="2401.18059_p008_b0126",
)

add_missing_table(
    "2401.18059", page=8,
    rows=[
        ["Model", "GPT-3 Acc.", "UnifiedQA Acc."],
        ["BM25", "57.3", "49.9"],
        ["DPR", "60.4", "53.9"],
        ["RAPTOR", "62.4", "56.6"],
    ],
    caption_block_id="2401.18059_p008_b0129",
)

add_missing_table(
    "2401.18059", page=8,
    rows=[
        ["Model", "F-1 Match"],
        ["LongT5 XL (Guo et al., 2022)", "53.1"],
        ["CoLT5 XL (Ainslie et al., 2023)", "53.9"],
        ["RAPTOR + GPT-4", "55.7"],
    ],
    caption_block_id="2401.18059_p008_b0132",
)

add_missing_table(
    "2401.18059", page=9,
    rows=[
        ["Model", "ROUGE-L", "BLEU-1", "BLEU-4", "METEOR"],
        ["BiDAF (Kočiskỳ et al., 2018)", "6.2", "5.7", "0.3", "3.7"],
        ["BM25 + BERT (Mou et al., 2020)", "15.5", "14.5", "1.4", "5.0"],
        ["Recursively Summarizing Books (Wu et al., 2021)", "21.6", "22.3", "4.2", "10.6"],
        ["Retriever + Reader (Izacard & Grave, 2022)", "32.0", "35.3", "7.5", "11.1"],
        ["RAPTOR + UnifiedQA", "30.8", "23.5", "6.4", "19.1"],
    ],
    caption_block_id="2401.18059_p009_b0429",
)

add_missing_table(
    "2401.18059", page=9,
    rows=[
        ["Model", "Accuracy Test Set", "Accuracy Hard Subset"],
        ["Longformer-base (Beltagy et al., 2020)", "39.5", "35.3"],
        ["DPR and DeBERTaV3-large (Pang et al., 2022)", "55.4", "46.1"],
        ["CoLISA (DeBERTaV3-large) (Dong et al., 2023a)", "62.3", "54.7"],
        ["RAPTOR + GPT-4", "82.6", "76.2"],
    ],
    caption_block_id="2401.18059_p009_b0431",
)

add_missing_table(
    "2401.18059", page=9,
    rows=[
        ["Layers Queried / Start Layer", "Layer 0 (Leaf Nodes)", "Layer 1", "Layer 2"],
        ["1 layer", "57.9", "57.8", "57.9"],
        ["2 layers", "-", "52.6", "63.15"],
        ["3 layers", "-", "-", "73.68"],
    ],
    caption_block_id="2401.18059_p009_b0430",
)

apply_fix("2403.14403", "2403.14403_p009_b0195", "2403.14403_p009_b0393")
apply_fix("2405.06211", "2405.06211_p007_b0262", "2405.06211_p007_b0639")
apply_fix("2312.10997", "2312.10997_p006_b0068", "2312.10997_p006_b0454")
apply_fix("2312.10997", "2312.10997_p013_b0173", "2312.10997_p013_b0455")
apply_fix("2401.17043", "2401.17043_p026_b0551", "2401.17043_p026_b0709")
apply_fix("2603.27355", "2603.27355_p012_b0176", "2603.27355_p012_b0309")
apply_fix("2311.09476", "2311.09476_p007_b0101", "2311.09476_p007_b0357")
apply_fix("2401.15884", "2401.15884_p007_b0131", "2401.15884_p007_b0322")
apply_fix("2603.27355", "2603.27355_p012_b0173", "2603.27355_p012_b0308")

### Manual data correction: garbled table header (2311.09476)
One extracted table header came through as parser junk (a stray section-number fragment) rather than the real column label. Fixed by direct field edit rather than a parser change, since this was a single-cell, single-paper issue — not worth a general-purpose rule. Recorded here (not silently) as part of the correction audit trail.

In [ ]:
data = load_paper("2311.09476")
table = next(b for b in data["blocks"] if b["block_id"] == "2311.09476_p007_b0357")
table["content"][0][0] = "WoW"  # replace the "5.2\nARES Performance on AIS" junk with the real header
save_paper("2311.09476", data)

### Corpus quality verification
Confirms the state referenced in project notes: zero remaining flags across all 30 papers after the correction pass above.

In [ ]:
import json, os

summary = []
for fname in os.listdir("parsed_output"):
    if not fname.endswith(".json") or fname.startswith("_"):
        continue
    with open(os.path.join("parsed_output", fname)) as f:
        data = json.load(f)
    blocks = data["blocks"]
    flagged = [b for b in blocks if b.get("needs_review")]
    summary.append({"paper_id": data["paper_id"], "num_blocks": len(blocks), "num_flagged": len(flagged)})

for s in sorted(summary, key=lambda x: x["paper_id"]):
    print(f"{s['paper_id']}: {s['num_blocks']} blocks, {s['num_flagged']} still flagged")

### Appendix: Camelot table-detection ceiling (RGB Tables 3/5/7)
Documented here as a confirmed hard limitation, not a bug to chase further. Camelot's `lattice` and `stream` flavors were both tested against RGB's dense two-column IEEE layout and neither reliably isolates the borderless, side-by-side tables on these pages — hence RGB's inclusion in the `manual_transcription_flag` set above.

In [ ]:
import requests
url = "https://arxiv.org/pdf/2309.01431"
open("rgb.pdf", "wb").write(requests.get(url).content)

import camelot
pages = "5,6,7"
for flavor in ("lattice", "stream"):
    found = camelot.read_pdf("rgb.pdf", pages=pages, flavor=flavor)
    print(f"\n{flavor}: {len(found)} raw tables")
    for t in found:
        acc = t.parsing_report.get("accuracy")
        bbox = t._bbox
        height = bbox[3] - bbox[1] if bbox else None
        rows = t.df.values.tolist()
        cell_lengths = [len(str(c)) for row in rows for c in row if str(c).strip()]
        avg_len = sum(cell_lengths) / len(cell_lengths) if cell_lengths else 0
        print(f"  page={t.page}, shape={t.df.shape}, accuracy={acc}, "
              f"bbox_height={height:.1f}, avg_cell_len={avg_len:.1f}")

# Conclusion: neither flavor cleanly separates the side-by-side tables on these pages
# (confirmed against higher line_scale, split table_areas, and pdfplumber's own
# extract_tables() -- same result across all approaches). Treated as a hard ceiling;
# RGB's tables are hand-transcribed instead (see manual_transcription_flag).

### Backup parsed output to Drive

In [ ]:
import shutil
shutil.copytree("parsed_output", "/content/drive/MyDrive/RAG/parsed_output")

import os
print(len(os.listdir("/content/drive/MyDrive/RAG/parsed_output")))

## Module 2: Benchmarking, Retrieval & Evaluation
Picks up after parsing/correction above. Environment, Drive mount, `scripts/` path, and the Gemini API key were already set up in **Environment setup** — no need to repeat them here.

### QA benchmark generation
Generates QA pairs via the Gemini API (`--resume` picks up from any existing `qa_pairs_raw.json` rather than overwriting it).

In [ ]:
!python scripts/generate_qa_pairs.py --resume

### Fix manual-transcription flags
One-off metadata correction: re-derives `manual_transcription_flag` for every QA pair from the current parsed data, in case a paper's flag status changed after a correction pass.

In [ ]:
!python scripts/fix_manual_transcription_flags.py

### QA verification workflow
Interactive review helpers — `load()` picks up any in-progress `qa_pairs_verified.json`, otherwise starts fresh from `qa_pairs_raw.json`.

In [ ]:
from verify_qa_pairs import *
load()
verification_report()

### Review loop (interactive — fill in `qa_id` per item)
Run `show_next_batch`, inspect each item, then call `approve("<qa_id>")`, `apply_fix("<qa_id>", question="<None>", answer="<None>", notes=None, gold_block_ids=None, paper_ids=None)` or `reject("<qa_id>", reason)` before `save()`. Repeat until `verification_report()` shows everything reviewed.

In [ ]:
show_next_batch(n=5)

In [ ]:
approve("")
save()

### Scope decision: cross-paper multi-hop discontinued
Cross-paper `multi_hop_comparative` items produced too many false connections from coincidental term/label overlap between unrelated papers, so this category is now within-paper only. All previously-generated cross-paper items are rejected here (logged, not silently deleted) so `--resume` won't regenerate them.

**Note:** this supersedes the ~65/35 within/cross-paper split noted earlier.

In [ ]:
import verify_qa_pairs as vqp
vqp.load()

cross_paper_ids = [item["qa_id"] for item in vqp._qa if item["category"] == "multi_hop_comparative" and item["cross_paper"]]
print(f"Removing {len(cross_paper_ids)} cross-paper item(s)")

for qa_id in cross_paper_ids:
    vqp.reject(qa_id, reason="cross-paper multi-hop discontinued as a category — too many false connections from coincidental term/label overlap between unrelated papers")

vqp.save()

### Fold verified set back into the working file
From here on, `qa_pairs_raw.json` holds the verified set (so downstream cells that reference it get the reviewed data).

In [ ]:
import shutil
shutil.copy("qa_pairs_verified.json", "qa_pairs_raw.json")

### De-duplicate by gold-block set
Checks for QA pairs that ended up pointing at the same `gold_block_ids` (can happen across resumed generation runs), then removes the duplicates.

In [ ]:
# Check for duplicates

import json
from collections import Counter

with open("qa_pairs_raw.json") as f:
    data = json.load(f)

block_counts = Counter(tuple(item["gold_block_ids"]) for item in data)
dupes = {k: v for k, v in block_counts.items() if v > 1}
print("Duplicate gold_block_ids used:", dupes)


In [ ]:
# deduplicate

import json

with open("qa_pairs_raw.json") as f:
    data = json.load(f)

seen_blocks = set()
deduped = []
removed = 0

for item in data:
    key = tuple(item["gold_block_ids"])
    if key in seen_blocks:
        removed += 1
        continue
    seen_blocks.add(key)
    deduped.append(item)

with open("qa_pairs_raw.json", "w") as f:
    json.dump(deduped, f, indent=2)

print(f"Removed {removed} duplicate(s). Remaining: {len(deduped)}")
print("Categories now:", {c: sum(1 for i in deduped if i["category"] == c) for c in set(i["category"] for i in deduped)})

In [ ]:
# Check final length after de-duplication

import json
with open("qa_pairs_raw.json") as f:
    data = json.load(f)
print(len(data))
print([d["qa_id"] for d in data])

### Main RAG pipeline
From here on: build both chunk sets (fixed-size and structure-aware), index each into its own Chroma collection, then run retrieval scoring and evaluation identically across both so the comparison is apples-to-apples.

**1. Fixed-size chunking** — 1000-char chunks with 50-char overlap, tracking each block's character span so gold blocks can later be mapped onto whichever chunk(s) they overlap.

In [ ]:
import json
import os

def build_fixed_size_chunks(paper, chunk_size=1000, overlap=50):
    blocks_sorted = sorted(paper['blocks'], key=lambda b: (b['page'], b['block_id']))

    full_text = ""
    block_spans = {}

    for b in blocks_sorted:
        content = b['content']
        if content is None:
          continue
        if isinstance(content, list):
            text = "\n".join(
                " | ".join(str(cell) for cell in row) for row in content
            )
        else:
            text = str(content)

        start = len(full_text)
        full_text += text + "\n\n"
        end = len(full_text)
        block_spans[b['block_id']] = (start, end)

    chunks = []
    pos = 0
    chunk_idx = 0
    while pos < len(full_text):
        chunk_end = min(pos + chunk_size, len(full_text))
        chunks.append({
            'chunk_id': f"{paper['paper_id']}_fixed_{chunk_idx:04d}",
            'paper_id': paper['paper_id'],
            'text': full_text[pos:chunk_end],
            'start_char': pos,
            'end_char': chunk_end
        })
        chunk_idx += 1
        if chunk_end == len(full_text):
            break
        pos += (chunk_size - overlap)

    return chunks, block_spans


def process_all_papers(parsed_dir, output_dir, chunk_size=1000, overlap=50):
    os.makedirs(output_dir, exist_ok=True)
    all_chunks = []
    all_block_spans = {}

    for fname in os.listdir(parsed_dir):
        if not fname.endswith('.json') or fname.startswith('_'):
            continue
        with open(os.path.join(parsed_dir, fname)) as f:
            paper = json.load(f)

        chunks, block_spans = build_fixed_size_chunks(paper, chunk_size, overlap)
        all_chunks.extend(chunks)
        all_block_spans[paper['paper_id']] = block_spans

    with open(os.path.join(output_dir, 'fixed_size_chunks.json'), 'w') as f:
        json.dump(all_chunks, f, indent=2)

    with open(os.path.join(output_dir, 'block_spans.json'), 'w') as f:
        json.dump(all_block_spans, f, indent=2)

    print(f"Total fixed-size chunks: {len(all_chunks)}")
    print(f"Papers processed: {len(all_block_spans)}")
    return all_chunks, all_block_spans



PARSED_DIR = '/content/drive/MyDrive/RAG/parsed_output/'
OUTPUT_DIR = '/content/drive/MyDrive/RAG/chunks/'

chunks, block_spans = process_all_papers(PARSED_DIR, OUTPUT_DIR)

**2. Gold-block → chunk mapping** — for each QA pair's `gold_block_ids`, finds every fixed-size chunk covering at least 50% of that block's character span (swept at 30%/70% too, for sensitivity). This is the fixed-size equivalent of `gold_block_ids`, since fixed-size chunks don't align with block boundaries the way structure-aware chunks do.

In [ ]:
import json

def compute_overlap_chars(a_start, a_end, b_start, b_end):
    return max(0, min(a_end, b_end) - max(a_start, b_start))


def build_gold_chunk_mapping(qa_pairs, fixed_chunks, block_spans, threshold=0.5):
    chunks_by_paper = {}
    for c in fixed_chunks:
        chunks_by_paper.setdefault(c['paper_id'], []).append(c)

    mapping = {}
    unmatched_blocks = []

    for qa in qa_pairs:
        relevant_chunks = set()

        for paper_id in qa['paper_ids']:
            paper_block_spans = block_spans.get(paper_id, {})
            paper_chunks = chunks_by_paper.get(paper_id, [])

            for block_id in qa['gold_block_ids']:
                if block_id not in paper_block_spans:
                    continue

                b_start, b_end = paper_block_spans[block_id]
                block_len = b_end - b_start
                if block_len == 0:
                    continue

                found_match = False
                for chunk in paper_chunks:
                    overlap = compute_overlap_chars(
                        b_start, b_end, chunk['start_char'], chunk['end_char']
                    )
                    coverage = overlap / block_len
                    if coverage >= threshold:
                        relevant_chunks.add(chunk['chunk_id'])
                        found_match = True

                if not found_match:
                    unmatched_blocks.append({
                        'qa_id': qa['qa_id'],
                        'block_id': block_id,
                        'paper_id': paper_id
                    })

        mapping[qa['qa_id']] = list(relevant_chunks)

    return mapping, unmatched_blocks


QA_PATH = '/content/drive/MyDrive/RAG/qa_pairs_verified.json'
CHUNKS_DIR = '/content/drive/MyDrive/RAG/chunks/'

with open(QA_PATH) as f:
    qa_pairs = json.load(f)
with open(CHUNKS_DIR + 'fixed_size_chunks.json') as f:
    fixed_chunks = json.load(f)
with open(CHUNKS_DIR + 'block_spans.json') as f:
    block_spans = json.load(f)

mapping, unmatched = build_gold_chunk_mapping(qa_pairs, fixed_chunks, block_spans, threshold=0.5)

with open(CHUNKS_DIR + 'gold_chunk_mapping_t50.json', 'w') as f:
    json.dump(mapping, f, indent=2)

print(f"QA pairs mapped: {len(mapping)}")
print(f"Unmatched gold blocks (no chunk met 50% threshold): {len(unmatched)}")
if unmatched:
    print("Sample unmatched:", unmatched[:13])

for t in [0.3, 0.7]:
    m, u = build_gold_chunk_mapping(qa_pairs, fixed_chunks, block_spans, threshold=t)
    with open(CHUNKS_DIR + f'gold_chunk_mapping_t{int(t*100)}.json', 'w') as f:
        json.dump(m, f, indent=2)
    print(f"Threshold {t}: {len(u)} unmatched blocks")

**3. Unmatched gold blocks — length check.** For blocks that never hit the 50% coverage threshold against any fixed-size chunk: how long is the block itself? (Very short blocks are the most likely to fall entirely within one chunk's overlap gap rather than genuinely being missed.)

In [ ]:
unmatched_block_ids = set(u['block_id'] for u in unmatched)  # from threshold=0.5 run

for paper_id, spans in block_spans.items():
    for block_id, (start, end) in spans.items():
        if block_id in unmatched_block_ids:
            print(f"{block_id}: length={end - start} chars")

**4. Structure-aware chunker + dual Chroma indexing** — one chunk per block (tables/figures/captions as atomic units), embedded and indexed alongside the fixed-size chunks so both live in the same Chroma DB under separate collections.

In [ ]:
!pip install -q chromadb google-genai

import json
import os
import time
import chromadb
from google import genai

# ---- Setup ----
PARSED_DIR = '/content/drive/MyDrive/RAG/parsed_output/'
CHUNKS_DIR = '/content/drive/MyDrive/RAG/chunks/'
CHROMA_DIR = '/content/drive/MyDrive/RAG/chroma_db/'

client_genai = genai.Client()
EMBED_MODEL = 'gemini-embedding-001'

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)


# ---- 1. Build structure-aware chunks (one per block_id) ----
def build_structure_aware_chunks(paper):
    chunks = []
    for b in paper['blocks']:
        content = b['content']
        if content is None:
          continue
        if isinstance(content, list):
            text = "\n".join(" | ".join(str(cell) for cell in row) for row in content)
        else:
            text = str(content)

        if not text.strip():
            continue

        chunks.append({
            'chunk_id': b['block_id'],
            'paper_id': paper['paper_id'],
            'block_type': b['block_type'],
            'text': text,
            'caption_of': b.get('caption_of'),
            'refers_to': b.get('refers_to'),
        })
    return chunks

def build_all_structure_aware_chunks(parsed_dir):
    all_chunks = []
    for fname in os.listdir(parsed_dir):
        if not fname.endswith('.json') or fname.startswith('_'):
            continue
        with open(os.path.join(parsed_dir, fname)) as f:
            paper = json.load(f)
        all_chunks.extend(build_structure_aware_chunks(paper))
    return all_chunks

structure_chunks_fixed = build_all_structure_aware_chunks(PARSED_DIR)
with open(CHUNKS_DIR + 'structure_aware_chunks.json', 'w') as f:
    json.dump(structure_chunks_fixed, f, indent=2)
print(f"New chunk count: {len(structure_chunks_fixed)}")


# ---- 2. Embedding helper (batched, with retry/backoff) ----
def embed_texts(texts, batch_size=20, max_retries=3):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        for attempt in range(max_retries):
            try:
                result = client_genai.models.embed_content(
                    model=EMBED_MODEL,
                    contents=batch,
                )
                embeddings.extend([e.values for e in result.embeddings])
                break
            except Exception as e:
                wait = 2 ** attempt
                print(f"Batch {i} failed (attempt {attempt+1}): {e}. Retrying in {wait}s...")
                time.sleep(wait)
        else:
            raise RuntimeError(f"Batch {i} failed after {max_retries} retries")
        time.sleep(0.3)
    return embeddings


# ---- 3. Index a chunk set into a Chroma collection ----
def index_chunks(collection_name, chunks, text_key='text', batch_size=100):
    def clean_metadata(c):
        return {
            k: (v if isinstance(v, (str, int, float, bool)) else json.dumps(v))
            for k, v in c.items() if k != text_key and v is not None
        }

    collection = chroma_client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )

    # Idempotent: skip chunks already present in this (persistent, Drive-backed) collection
    # so re-running this cell in a fresh runtime doesn't re-embed/re-add everything and
    # burn API calls on chunks that were already indexed in a prior session.
    all_ids = [c['chunk_id'] for c in chunks]
    existing_ids = set(collection.get(ids=all_ids)['ids']) if collection.count() else set()
    chunks = [c for c in chunks if c['chunk_id'] not in existing_ids]
    if not chunks:
        print(f"'{collection_name}': all {len(all_ids)} chunks already indexed, nothing to do")
        return collection
    print(f"'{collection_name}': {len(existing_ids)} already indexed, embedding {len(chunks)} new chunks")

    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        texts = [c[text_key] for c in batch]
        ids = [c['chunk_id'] for c in batch]
        metadatas = [clean_metadata(c) for c in batch]

        embeddings = embed_texts(texts)

        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas,
        )
        print(f"  Indexed {i + len(batch)}/{len(chunks)} into '{collection_name}'")

    print(f"Done: '{collection_name}' has {collection.count()} items")
    return collection

print("\n--- Indexing structure-aware collection ---")
structure_collection = index_chunks('structure_aware', structure_chunks_fixed)

print("\n--- Indexing fixed-size collection (chunk_size=1000) ---")
with open(CHUNKS_DIR + 'fixed_size_chunks.json') as f:
    fixed_chunks = json.load(f)
fixed_collection = index_chunks('fixed_size_1000', fixed_chunks)

Reset utility (kept commented — only needed if re-indexing from scratch).

In [ ]:
#chroma_client.delete_collection('structure_aware')
#chroma_client.delete_collection('fixed_size_1000')

**5. Retrieval + Recall@k/MRR scoring** — embeds every benchmark question, queries both collections at top-k=10, and scores Recall@5, Recall@10, and MRR against each chunking scheme's own gold set.

In [ ]:
import json
import time
import chromadb
from google import genai
import os
from google.colab import userdata

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
client_genai = genai.Client()
EMBED_MODEL = 'gemini-embedding-001'

CHUNKS_DIR = '/content/drive/MyDrive/RAG/chunks/'
CHROMA_DIR = '/content/drive/MyDrive/RAG/chroma_db/'
QA_PATH = '/content/drive/MyDrive/RAG/qa_pairs_verified.json'

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
structure_collection = chroma_client.get_collection('structure_aware')
fixed_collection = chroma_client.get_collection('fixed_size_1000')

with open(QA_PATH) as f:
    qa_pairs = json.load(f)
with open(CHUNKS_DIR + 'gold_chunk_mapping_t50.json') as f:
    fixed_gold_mapping = json.load(f)


# ---- 1. Embed all questions (batched) ----
def embed_texts(texts, batch_size=20, max_retries=3, sleep=0.3):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        for attempt in range(max_retries):
            try:
                result = client_genai.models.embed_content(model=EMBED_MODEL, contents=batch)
                embeddings.extend([e.values for e in result.embeddings])
                break
            except Exception as e:
                wait = 2 ** attempt
                print(f"Batch {i} failed (attempt {attempt+1}): {e}. Retrying in {wait}s...")
                time.sleep(wait)
        else:
            raise RuntimeError(f"Batch {i} failed after {max_retries} retries")
        time.sleep(sleep)
    return embeddings


# Cache question embeddings to disk (keyed by qa_id) so re-running this cell in a fresh
# runtime reuses embeddings from a prior session instead of re-calling the API for
# questions we've already embedded.
QUESTION_EMB_CACHE_PATH = CHUNKS_DIR + 'question_embeddings_cache.json'
if os.path.exists(QUESTION_EMB_CACHE_PATH):
    with open(QUESTION_EMB_CACHE_PATH) as f:
        question_embedding_cache = json.load(f)
else:
    question_embedding_cache = {}

qa_ids_ordered = [qa['qa_id'] for qa in qa_pairs]
missing = [(qid, qa['question']) for qid, qa in zip(qa_ids_ordered, qa_pairs) if qid not in question_embedding_cache]

if missing:
    print(f"Embedding {len(missing)} new questions ({len(qa_ids_ordered) - len(missing)} already cached)")
    new_embeddings = embed_texts([q for _, q in missing])
    for (qid, _), emb in zip(missing, new_embeddings):
        question_embedding_cache[qid] = emb
    with open(QUESTION_EMB_CACHE_PATH, 'w') as f:
        json.dump(question_embedding_cache, f)
else:
    print("All question embeddings already cached, no API calls needed")

question_embeddings = [question_embedding_cache[qid] for qid in qa_ids_ordered]
print(f"Using {len(question_embeddings)} question embeddings")


# ---- 2. Query both collections at top-k=10 (score at k=5 and k=10 from same result) ----
MAX_K = 10

def query_collection(collection, embedding, k=MAX_K):
    result = collection.query(query_embeddings=[embedding], n_results=k)
    return result['ids'][0]


# ---- 3. Scoring functions ----
def recall_at_k(ranked_ids, relevant_set, k):
    return 1 if any(cid in relevant_set for cid in ranked_ids[:k]) else 0

def reciprocal_rank(ranked_ids, relevant_set):
    for rank, cid in enumerate(ranked_ids, start=1):
        if cid in relevant_set:
            return 1.0 / rank
    return 0.0


# ---- 4. Run retrieval + scoring per QA pair ----
results = []

for qa, q_emb in zip(qa_pairs, question_embeddings):
    qa_id = qa['qa_id']

    # Gold sets
    structure_gold = set(qa['gold_block_ids'])  # direct block_id match
    fixed_gold = set(fixed_gold_mapping.get(qa_id, []))

    # Retrieve
    structure_ranked = query_collection(structure_collection, q_emb)
    fixed_ranked = query_collection(fixed_collection, q_emb)

    row = {
        'qa_id': qa_id,
        'category': qa['category'],
        'cross_paper': qa['cross_paper'],
        'manual_transcription_flag': qa['manual_transcription_flag'],
        'structure_recall@5': recall_at_k(structure_ranked, structure_gold, 5),
        'structure_recall@10': recall_at_k(structure_ranked, structure_gold, 10),
        'structure_mrr': reciprocal_rank(structure_ranked, structure_gold),
        'fixed_recall@5': recall_at_k(fixed_ranked, fixed_gold, 5),
        'fixed_recall@10': recall_at_k(fixed_ranked, fixed_gold, 10),
        'fixed_mrr': reciprocal_rank(fixed_ranked, fixed_gold),
    }
    results.append(row)

with open(CHUNKS_DIR + 'retrieval_scores_per_qa.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Scored {len(results)} QA pairs")


# ---- 5. Aggregate: overall + by category ----
import pandas as pd

df = pd.DataFrame(results)

def summarize(subdf, label):
    print(f"\n--- {label} (n={len(subdf)}) ---")
    for col in ['structure_recall@5', 'structure_recall@10', 'structure_mrr',
                'fixed_recall@5', 'fixed_recall@10', 'fixed_mrr']:
        print(f"  {col}: {subdf[col].mean():.3f}")

summarize(df, "OVERALL")
for cat in df['category'].unique():
    summarize(df[df['category'] == cat], cat)

df.to_csv(CHUNKS_DIR + 'retrieval_scores_summary.csv', index=False)

**6. Error analysis** — qualitative look at cases where one chunking scheme recalls the gold block(s) and the other doesn't, plus a look at the pairs where *both* fail.

In [ ]:
# --- Error analysis: pull qualitative failure examples ---

import json

CHUNKS_DIR = '/content/drive/MyDrive/RAG/chunks/'

with open(CHUNKS_DIR + 'retrieval_scores_per_qa.json') as f:
    results = json.load(f)
with open('/content/drive/MyDrive/RAG/qa_pairs_verified.json') as f:
    qa_pairs = json.load(f)

qa_lookup = {qa['qa_id']: qa for qa in qa_pairs}

# Case 1: structure-aware succeeded, fixed-size failed
structure_wins = [
    r for r in results
    if r['structure_recall@10'] == 1 and r['fixed_recall@10'] == 0
]

# Case 2: fixed-size succeeded, structure-aware failed
fixed_wins = [
    r for r in results
    if r['fixed_recall@10'] == 1 and r['structure_recall@10'] == 0
]

# Case 3: both failed (hardest QA pairs)
both_fail = [
    r for r in results
    if r['structure_recall@10'] == 0 and r['fixed_recall@10'] == 0
]

print(f"Structure-aware wins (fixed fails): {len(structure_wins)}")
print(f"Fixed-size wins (structure fails): {len(fixed_wins)}")
print(f"Both fail: {len(both_fail)}")

def show_examples(rows, label, n=5):
    print(f"\n=== {label} (showing up to {n} of {len(rows)}) ===")
    for r in rows[:n]:
        qa = qa_lookup[r['qa_id']]
        print(f"\nqa_id: {r['qa_id']} | category: {r['category']}")
        print(f"  question: {qa['question']}")
        print(f"  gold_block_ids: {qa['gold_block_ids']}")
        print(f"  paper_ids: {qa['paper_ids']}")

show_examples(structure_wins, "STRUCTURE-AWARE WINS")
show_examples(fixed_wins, "FIXED-SIZE WINS")
show_examples(both_fail, "BOTH FAIL")

# Save full lists for later reference in the report
with open(CHUNKS_DIR + 'failure_analysis.json', 'w') as f:
    json.dump({
        'structure_wins': structure_wins,
        'fixed_wins': fixed_wins,
        'both_fail': both_fail,
    }, f, indent=2)

Follow-up on the `both_fail` cases: how long is the gold block text itself? (Short blocks are more likely to be genuinely hard to retrieve regardless of chunking scheme.)

In [ ]:
# Diagnostic block

# For each "both_fail" QA pair, check the gold block's actual text length/specificity
with open(CHUNKS_DIR + 'structure_aware_chunks.json') as f:
    structure_chunks = {c['chunk_id']: c for c in json.load(f)}

for r in both_fail:
    qa = qa_lookup[r['qa_id']]
    for bid in qa['gold_block_ids']:
        chunk = structure_chunks.get(bid)
        if chunk:
            print(f"\n{r['qa_id']} | {r['category']}")
            print(f"  Q: {qa['question']}")
            print(f"  Gold block text (first 200 chars): {chunk['text'][:200]}")

**7. Statistical significance** — McNemar's test on binary Recall@k, Wilcoxon signed-rank on MRR, both overall and per-category.

In [ ]:
from scipy.stats import wilcoxon
import numpy as np

with open(CHUNKS_DIR + 'retrieval_scores_per_qa.json') as f:
    results = json.load(f)

structure_recall10 = [r['structure_recall@10'] for r in results]
fixed_recall10 = [r['fixed_recall@10'] for r in results]
structure_mrr = [r['structure_mrr'] for r in results]
fixed_mrr = [r['fixed_mrr'] for r in results]

# McNemar's test for paired binary outcomes (Recall@10)
from statsmodels.stats.contingency_tables import mcnemar
import pandas as pd

df = pd.DataFrame(results)
b = ((df['structure_recall@10'] == 1) & (df['fixed_recall@10'] == 0)).sum()  # structure wins
c = ((df['structure_recall@10'] == 0) & (df['fixed_recall@10'] == 1)).sum()  # fixed wins

table = [[0, b], [c, 0]]
result = mcnemar(table, exact=True)
print(f"McNemar's test (Recall@10): statistic={result.statistic}, p-value={result.pvalue:.6f}")
print(f"  Structure-only wins: {b}, Fixed-only wins: {c}")

# Wilcoxon signed-rank test for MRR (continuous paired scores)
stat, p = wilcoxon(structure_mrr, fixed_mrr)
print(f"\nWilcoxon signed-rank test (MRR): statistic={stat}, p-value={p:.6f}")

# Same tests per category
for cat in df['category'].unique():
    subdf = df[df['category'] == cat]
    b_cat = ((subdf['structure_recall@10'] == 1) & (subdf['fixed_recall@10'] == 0)).sum()
    c_cat = ((subdf['structure_recall@10'] == 0) & (subdf['fixed_recall@10'] == 1)).sum()
    if b_cat + c_cat > 0:
        res_cat = mcnemar([[0, b_cat], [c_cat, 0]], exact=True)
        print(f"\n{cat}: McNemar p={res_cat.pvalue:.4f} (structure wins={b_cat}, fixed wins={c_cat})")

**8. LLM-as-judge** — generate answers from each chunking scheme's retrieved context, judge each on correctness/completeness/faithfulness, then aggregate.

In [ ]:
import json
import time
import os
from google import genai

client_genai = genai.Client()
GEN_MODEL = 'gemini-3.5-flash'

CHUNKS_DIR = '/content/drive/MyDrive/RAG/chunks/'
CHECKPOINT_PATH = CHUNKS_DIR + 'generated_answers.json'

with open('/content/drive/MyDrive/RAG/qa_pairs_verified.json') as f:
    qa_pairs = json.load(f)
with open(CHUNKS_DIR + 'structure_aware_chunks.json') as f:
    structure_chunks = {c['chunk_id']: c for c in json.load(f)}
with open(CHUNKS_DIR + 'fixed_size_chunks.json') as f:
    fixed_chunks = {c['chunk_id']: c for c in json.load(f)}

import chromadb
chroma_client = chromadb.PersistentClient(path='/content/drive/MyDrive/RAG/chroma_db/')
structure_collection = chroma_client.get_collection('structure_aware')
fixed_collection = chroma_client.get_collection('fixed_size_1000')

# ---- Load existing checkpoint if present ----
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        generated_answers = json.load(f)
    done_ids = {item['qa_id'] for item in generated_answers}
    print(f"Resuming: {len(done_ids)} already generated")
else:
    generated_answers = []
    done_ids = set()


def build_prompt(question, context_texts):
    context = "\n\n---\n\n".join(context_texts)
    return f"""Answer the question using ONLY the context provided below. Be concise and specific. If the context does not contain enough information to answer, say "Cannot be determined from the provided context."

Context:
{context}

Question: {question}

Answer:"""


def generate_answer(question, context_texts, max_retries=3):
    prompt = build_prompt(question, context_texts)
    for attempt in range(max_retries):
        try:
            response = client_genai.models.generate_content(model=GEN_MODEL, contents=prompt)
            return response.text.strip()
        except Exception as e:
            wait = 2 ** attempt
            print(f"Generation failed (attempt {attempt+1}): {e}. Retrying in {wait}s...")
            time.sleep(wait)
    return "GENERATION_FAILED"


def get_top_k_texts(chunk_lookup, ranked_ids, k=5):
    return [chunk_lookup[cid]['text'] for cid in ranked_ids[:k] if cid in chunk_lookup]


CHECKPOINT_EVERY = 10

for idx, qa in enumerate(qa_pairs):
    qa_id = qa['qa_id']
    if qa_id in done_ids:
        continue

    question = qa['question']

    q_emb = client_genai.models.embed_content(
        model='gemini-embedding-001', contents=[question]
    ).embeddings[0].values
    time.sleep(0.3)

    structure_ranked = structure_collection.query(query_embeddings=[q_emb], n_results=5)['ids'][0]
    fixed_ranked = fixed_collection.query(query_embeddings=[q_emb], n_results=5)['ids'][0]

    structure_context = get_top_k_texts(structure_chunks, structure_ranked, k=5)
    fixed_context = get_top_k_texts(fixed_chunks, fixed_ranked, k=5)

    structure_answer = generate_answer(question, structure_context)
    time.sleep(0.5)
    fixed_answer = generate_answer(question, fixed_context)
    time.sleep(0.5)

    generated_answers.append({
        'qa_id': qa_id,
        'category': qa['category'],
        'question': question,
        'gold_answer': qa['answer'],
        'structure_answer': structure_answer,
        'fixed_answer': fixed_answer,
    })
    print(f"Generated: {qa_id} ({len(generated_answers)}/{len(qa_pairs)})")

    if len(generated_answers) % CHECKPOINT_EVERY == 0:
        with open(CHECKPOINT_PATH, 'w') as f:
            json.dump(generated_answers, f, indent=2)
        print(f"  [checkpoint saved at {len(generated_answers)}]")

# final save
with open(CHECKPOINT_PATH, 'w') as f:
    json.dump(generated_answers, f, indent=2)
print(f"\nDone. Total generated: {len(generated_answers)}")

In [ ]:
import json
import time
import os
from google import genai
from google.colab import userdata

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
client_genai = genai.Client()

CHUNKS_DIR = '/content/drive/MyDrive/RAG/chunks/'
JUDGE_MODEL = 'gemini-3.1-pro-preview'
JUDGE_CHECKPOINT_PATH = CHUNKS_DIR + 'judged_answers.json'

with open(CHUNKS_DIR + 'generated_answers.json') as f:
    generated_answers = json.load(f)

if os.path.exists(JUDGE_CHECKPOINT_PATH):
    with open(JUDGE_CHECKPOINT_PATH) as f:
        judged_results = json.load(f)
    judged_done_ids = {item['qa_id'] for item in judged_results}
    print(f"Resuming: {len(judged_done_ids)} already judged")
else:
    judged_results = []
    judged_done_ids = set()

JUDGE_PROMPT_TEMPLATE = """You are evaluating the quality of an AI-generated answer against a gold reference answer.

Question: {question}

Gold reference answer: {gold_answer}

Generated answer: {generated_answer}

Rate the generated answer on three dimensions, each on a scale of 1-5:
- Correctness: Does it factually match the gold answer? (1=completely wrong, 5=fully correct)
- Completeness: Does it cover all key information in the gold answer? (1=missing everything, 5=fully complete)
- Faithfulness: Is it free of hallucinated/unsupported claims? (1=heavily hallucinated, 5=fully grounded)

Respond ONLY with a JSON object in this exact format, nothing else:
{{"correctness": <int>, "completeness": <int>, "faithfulness": <int>, "rationale": "<one sentence>"}}"""


def judge_answer(question, gold_answer, generated_answer, max_retries=3):
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        question=question, gold_answer=gold_answer, generated_answer=generated_answer
    )
    for attempt in range(max_retries):
        try:
            response = client_genai.models.generate_content(model=JUDGE_MODEL, contents=prompt)
            text = response.text.strip().replace('```json', '').replace('```', '').strip()
            return json.loads(text)
        except Exception as e:
            wait = 2 ** attempt
            print(f"Judge call failed (attempt {attempt+1}): {e}. Retrying in {wait}s...")
            time.sleep(wait)
    return {"correctness": None, "completeness": None, "faithfulness": None, "rationale": "JUDGE_FAILED"}


CHECKPOINT_EVERY = 10

for item in generated_answers:
    qa_id = item['qa_id']
    if qa_id in judged_done_ids:
        continue

    structure_scores = judge_answer(item['question'], item['gold_answer'], item['structure_answer'])
    time.sleep(0.5)
    fixed_scores = judge_answer(item['question'], item['gold_answer'], item['fixed_answer'])
    time.sleep(0.5)

    judged_results.append({
        'qa_id': qa_id,
        'category': item['category'],
        'structure_scores': structure_scores,
        'fixed_scores': fixed_scores,
    })
    print(f"Judged: {qa_id} ({len(judged_results)}/{len(generated_answers)})")

    if len(judged_results) % CHECKPOINT_EVERY == 0:
        with open(JUDGE_CHECKPOINT_PATH, 'w') as f:
            json.dump(judged_results, f, indent=2)
        print(f"  [checkpoint saved at {len(judged_results)}]")

with open(JUDGE_CHECKPOINT_PATH, 'w') as f:
    json.dump(judged_results, f, indent=2)
print(f"\nDone. Total judged: {len(judged_results)}")

In [ ]:
import json
import pandas as pd

CHUNKS_DIR = '/content/drive/MyDrive/RAG/chunks/'

with open(CHUNKS_DIR + 'judged_answers.json') as f:
    judged_results = json.load(f)

rows = []
for r in judged_results:
    rows.append({
        'qa_id': r['qa_id'],
        'category': r['category'],
        'structure_correctness': r['structure_scores']['correctness'],
        'structure_completeness': r['structure_scores']['completeness'],
        'structure_faithfulness': r['structure_scores']['faithfulness'],
        'fixed_correctness': r['fixed_scores']['correctness'],
        'fixed_completeness': r['fixed_scores']['completeness'],
        'fixed_faithfulness': r['fixed_scores']['faithfulness'],
    })

df_judge = pd.DataFrame(rows)

print("--- OVERALL ---")
for col in ['structure_correctness', 'structure_completeness', 'structure_faithfulness',
            'fixed_correctness', 'fixed_completeness', 'fixed_faithfulness']:
    print(f"  {col}: {df_judge[col].mean():.2f}")

for cat in df_judge['category'].unique():
    print(f"\n--- {cat} ---")
    sub = df_judge[df_judge['category'] == cat]
    for col in ['structure_correctness', 'fixed_correctness']:
        print(f"  {col}: {sub[col].mean():.2f}")

df_judge.to_csv(CHUNKS_DIR + 'judged_scores_summary.csv', index=False)

### Evaluate low-scoring multi-hop rationales (optional)

This is done to understand the failure pattern in multi-hop pairs and compute a 'joint coverage' metric to identify the bottleneck for multi-hop answer quality.

A statistical significance test is run on the judge scores to confirm whether the near-tie in multi-hop is genuinely a tie or just underpowered at n=31.

In [ ]:
# identify multi-hop pairs' failure pattern (sanity check -- optional)

import json

CHUNKS_DIR = '/content/drive/MyDrive/RAG/chunks/'

with open(CHUNKS_DIR + 'judged_answers.json') as f:
    judged_results = json.load(f)
with open(CHUNKS_DIR + 'generated_answers.json') as f:
    generated_answers = {a['qa_id']: a for a in json.load(f)}

multi_hop_judged = [r for r in judged_results if r['category'] == 'multi_hop_comparative']

# Sort by structure_correctness ascending to surface the worst cases first
multi_hop_judged_sorted = sorted(
    multi_hop_judged,
    key=lambda r: (r['structure_scores'].get('correctness') or 0)
)

print(f"Total multi_hop_comparative judged: {len(multi_hop_judged)}\n")

for r in multi_hop_judged_sorted[:8]:  # worst 8
    qa_id = r['qa_id']
    gen = generated_answers[qa_id]
    print(f"=== {qa_id} ===")
    print(f"Question: {gen['question']}")
    print(f"Gold answer: {gen['gold_answer']}")
    print(f"\n--- Structure-aware (score: {r['structure_scores'].get('correctness')}) ---")
    print(f"Generated: {gen['structure_answer']}")
    print(f"Rationale: {r['structure_scores'].get('rationale')}")
    print(f"\n--- Fixed-size (score: {r['fixed_scores'].get('correctness')}) ---")
    print(f"Generated: {gen['fixed_answer']}")
    print(f"Rationale: {r['fixed_scores'].get('rationale')}")
    print("\n" + "="*80 + "\n")

In [ ]:
# Build a joint-coverage metric (sanity check -- optional)
!pip install -q chromadb google-genai

import json
import os
from google import genai
from google.colab import userdata
import chromadb

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
client_genai = genai.Client()

CHUNKS_DIR = '/content/drive/MyDrive/RAG/chunks/'
CHROMA_DIR = '/content/drive/MyDrive/RAG/chroma_db/'

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
structure_collection = chroma_client.get_collection('structure_aware')
fixed_collection = chroma_client.get_collection('fixed_size_1000')

with open('/content/drive/MyDrive/RAG/qa_pairs_verified.json') as f:
    qa_pairs = json.load(f)
with open(CHUNKS_DIR + 'gold_chunk_mapping_t50.json') as f:
    fixed_gold_mapping = json.load(f)

# Reuse question embeddings cached during the retrieval-scoring step instead of
# re-calling the embedding API for questions we've already embedded.
QUESTION_EMB_CACHE_PATH = CHUNKS_DIR + 'question_embeddings_cache.json'
if os.path.exists(QUESTION_EMB_CACHE_PATH):
    with open(QUESTION_EMB_CACHE_PATH) as f:
        question_embedding_cache = json.load(f)
else:
    question_embedding_cache = {}

# ---- Re-derive: did top-5 retrieval contain ALL gold blocks for multi-hop pairs? ----
multi_hop_qa = [qa for qa in qa_pairs if qa['category'] == 'multi_hop_comparative']

joint_coverage_structure = []
joint_coverage_fixed = []

for qa in multi_hop_qa:
    qa_id = qa['qa_id']
    gold_ids = set(qa['gold_block_ids'])
    if len(gold_ids) < 2:
        continue

    q_emb = question_embedding_cache.get(qa_id)
    if q_emb is None:  # only hits the API for questions missing from the cache
        q_emb = client_genai.models.embed_content(
            model='gemini-embedding-001', contents=[qa['question']]
        ).embeddings[0].values

    structure_top5 = set(structure_collection.query(query_embeddings=[q_emb], n_results=5)['ids'][0])
    fixed_top5 = set(fixed_collection.query(query_embeddings=[q_emb], n_results=5)['ids'][0])
    fixed_gold = set(fixed_gold_mapping.get(qa_id, []))

    joint_coverage_structure.append(gold_ids.issubset(structure_top5))
    joint_coverage_fixed.append(fixed_gold.issubset(fixed_top5) if fixed_gold else False)

print(f"Multi-hop pairs with 2+ gold blocks: {len(joint_coverage_structure)}")
print(f"Structure-aware: ALL gold blocks in top-5: {sum(joint_coverage_structure)}/{len(joint_coverage_structure)}")
print(f"Fixed-size: ALL gold blocks in top-5: {sum(joint_coverage_fixed)}/{len(joint_coverage_fixed)}")

In [ ]:
# Wilcoxon signed rank test for judge scores

import json
from scipy.stats import wilcoxon
import pandas as pd

CHUNKS_DIR = '/content/drive/MyDrive/RAG/chunks/'

with open(CHUNKS_DIR + 'judged_answers.json') as f:
    judged_results = json.load(f)

rows = []
for r in judged_results:
    rows.append({
        'qa_id': r['qa_id'],
        'category': r['category'],
        'structure_correctness': r['structure_scores'].get('correctness'),
        'structure_completeness': r['structure_scores'].get('completeness'),
        'structure_faithfulness': r['structure_scores'].get('faithfulness'),
        'fixed_correctness': r['fixed_scores'].get('correctness'),
        'fixed_completeness': r['fixed_scores'].get('completeness'),
        'fixed_faithfulness': r['fixed_scores'].get('faithfulness'),
    })

df = pd.DataFrame(rows).dropna()
print(f"Valid judged pairs (after dropping failures): {len(df)}")

# ---- Overall significance tests ----
def run_wilcoxon(df, col_a, col_b, label):
    diffs = df[col_a] - df[col_b]
    if (diffs == 0).all():
        print(f"{label}: no variation, cannot test")
        return
    stat, p = wilcoxon(df[col_a], df[col_b])
    print(f"{label}: statistic={stat:.1f}, p-value={p:.6f}, "
          f"mean_structure={df[col_a].mean():.2f}, mean_fixed={df[col_b].mean():.2f}")

print("\n--- OVERALL ---")
run_wilcoxon(df, 'structure_correctness', 'fixed_correctness', 'Correctness')
run_wilcoxon(df, 'structure_completeness', 'fixed_completeness', 'Completeness')
run_wilcoxon(df, 'structure_faithfulness', 'fixed_faithfulness', 'Faithfulness')

# ---- Per-category ----
for cat in df['category'].unique():
    sub = df[df['category'] == cat]
    print(f"\n--- {cat} (n={len(sub)}) ---")
    run_wilcoxon(sub, 'structure_correctness', 'fixed_correctness', 'Correctness')
    run_wilcoxon(sub, 'structure_completeness', 'fixed_completeness', 'Completeness')